# RACE Project — Experiments Notebook
Hyperparameter search, feature ablation, and qualitative samples.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, f1_score

PROCESSED = ROOT / 'data' / 'processed'
MODEL_A_TRAD = ROOT / 'models' / 'model_a' / 'traditional'
X_ohe_train = sp.load_npz(PROCESSED / 'X_ohe_train.npz')
X_ohe_val   = sp.load_npz(PROCESSED / 'X_ohe_val.npz')
X_comb_train = sp.load_npz(PROCESSED / 'X_combined_train.npz')
X_comb_val   = sp.load_npz(PROCESSED / 'X_combined_val.npz')
y_train = np.load(PROCESSED / 'y_train.npy')
y_val   = np.load(PROCESSED / 'y_val.npy')
print('Loaded:', X_ohe_train.shape, X_comb_train.shape, y_train.shape)

## 1. Hyperparameter Search — Logistic Regression `C`

In [ ]:
param_grid = {'C': [0.01, 0.1, 1.0, 10.0]}
lr = LogisticRegression(max_iter=500, solver='liblinear', class_weight='balanced', random_state=42)
gs = GridSearchCV(lr, param_grid, scoring='f1_macro', cv=3, n_jobs=-1)
gs.fit(X_ohe_train[:20000], y_train[:20000])
print('Best params:', gs.best_params_, 'best CV f1_macro:', round(gs.best_score_, 4))

## 2. Hyperparameter Search — SVM `C`

In [ ]:
svm_grid = {'C': [0.1, 1.0, 10.0]}
svc = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
gs2 = GridSearchCV(svc, svm_grid, scoring='f1_macro', cv=3, n_jobs=-1)
gs2.fit(X_ohe_train[:20000], y_train[:20000])
print('Best params:', gs2.best_params_, 'best CV f1_macro:', round(gs2.best_score_, 4))

## 3. Feature Ablation — OHE vs Lexical vs Combined

In [ ]:
lex_dim = X_comb_train.shape[1] - X_ohe_train.shape[1]
X_lex_train = X_comb_train[:, X_ohe_train.shape[1]:]
X_lex_val = X_comb_val[:, X_ohe_train.shape[1]:]
results = []
for name, Xtr, Xva in [
    ('OHE only',      X_ohe_train, X_ohe_val),
    ('Lexical only',  X_lex_train, X_lex_val),
    ('Combined',      X_comb_train, X_comb_val),
]:
    clf = LogisticRegression(max_iter=500, solver='liblinear', class_weight='balanced', random_state=42)
    clf.fit(Xtr, y_train)
    yp = clf.predict(Xva)
    results.append({'features': name,
                    'accuracy': round(accuracy_score(y_val, yp), 4),
                    'macro_f1': round(f1_score(y_val, yp, average='macro'), 4)})
pd.DataFrame(results)

## 4. K-Means k Sweep — Silhouette vs k

In [ ]:
rng = np.random.default_rng(42)
n = min(5000, X_ohe_train.shape[0])
idx = rng.choice(X_ohe_train.shape[0], size=n, replace=False)
X_sub = X_ohe_train[idx]
ks, sils = [], []
for k in [2, 4, 8, 16]:
    km = KMeans(n_clusters=k, n_init=5, random_state=42)
    labels = km.fit_predict(X_sub)
    sil = silhouette_score(X_sub, labels, metric='cosine')
    ks.append(k); sils.append(sil)
    print(f'  k={k}  silhouette={sil:.4f}')
plt.figure(figsize=(7, 4))
plt.plot(ks, sils, marker='o')
plt.xlabel('k'); plt.ylabel('Silhouette (cosine)')
plt.title('K-Means: silhouette vs k')
plt.grid(True); plt.show()

## 5. Distractor Quality — 10 Random Samples

In [ ]:
from src.inference import generate_distractors
test_df = pd.read_csv(PROCESSED / 'test.csv').sample(10, random_state=42).reset_index(drop=True)
rows = []
for _, r in test_df.iterrows():
    correct_label = str(r['answer']).strip().upper()
    answer_text = str(r.get(correct_label, ''))
    try:
        d = generate_distractors(r['article'], r['question'], answer_text)
    except Exception as e:
        d = [f'<error: {e}>']
    rows.append({'snippet': str(r['article'])[:80] + '…',
                 'correct': answer_text, 'd1': d[0] if len(d) > 0 else '',
                 'd2': d[1] if len(d) > 1 else '', 'd3': d[2] if len(d) > 2 else ''})
pd.DataFrame(rows)

## 6. BLEU / ROUGE Sample — 50 Generated vs RACE Original Questions

In [ ]:
from src.inference import generate_question
from src.evaluate_nlp import compute_bleu, compute_rouge_l
sample = pd.read_csv(PROCESSED / 'test.csv').head(50).reset_index(drop=True)
hyps, refs = [], []
for _, r in sample.iterrows():
    correct_label = str(r['answer']).strip().upper()
    ans = str(r.get(correct_label, ''))
    if not ans or not str(r['article']).strip():
        continue
    try:
        hyps.append(generate_question(r['article'], ans))
        refs.append(str(r['question']))
    except Exception:
        continue
print('BLEU:', compute_bleu(hyps, refs))
print('ROUGE-L:', compute_rouge_l(hyps, refs))